# レッスン 07 - プランニングデザインパターン

このノートブックでは、Microsoft Agent Framework を使用した AI エージェントの<strong>プランニングデザインパターン</strong>を示します。
複雑な旅行のリクエストを構造化されたサブタスクに分解し、
それらを専門のエージェントに割り当て、結果として得られた計画を実行する方法を学びます — すべては Pydantic モデルによる構造化出力を使って行います。


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## タスク分解

タスク分解はプランニングデザインパターンの核心です。複雑な要求を単一のエージェントに
エンドツーエンドで処理させるのではなく、問題を小さく明確な<strong>サブタスク</strong>に分割します。
各サブタスクは専門のエージェント（例：フライト、ホテル、アクティビティ）に割り当てられ、
優先度や依存関係の順序が明確に定義されます。

このアプローチにはいくつかの利点があります：
- <strong>明確さ</strong>：各サブタスクには単一の責任があります。
- <strong>並列性</strong>：独立したサブタスクは同時に実行できます。
- <strong>信頼性</strong>：失敗は個別のサブタスクに限定されます。
- <strong>予算追跡</strong>：コストはサブタスクごとに見積もられ集計されます。


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## 構造化出力を用いた計画エージェントの作成

計画エージェントは<strong>フロントデスクのコーディネーター</strong>として機能します。高レベルの旅行リクエストに対して、
構造化された `TravelPlan` を生成します。リクエストをサブタスクに分解し、優先順位を設定し、
コンシェルジュや実行層が作業を実施できるように依存関係を特定します。


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## 専門ツールを使ったプランの実行

フロントデスク担当者が構造化されたプランを作成したら、<strong>コンシェルジュエージェント</strong>がそれを実行します。
各専門ツールは、サブタスクの1つのカテゴリ（フライト、ホテル、アクティビティ）を担当します。コンシェルジュは
プランのサブタスクを依存順に反復処理し、それぞれを適切なツールに割り当てます。


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## 要約

このレッスンでは、AIエージェントのための **Planning Design Pattern** を学習しました：

1. <strong>タスク分解</strong> — フロントデスクのプランニングエージェントが複雑な旅行リクエストを
   Pydanticモデルを使用して構造化されたサブタスクに分解し、専門エージェントに優先順位と依存関係を
   割り当てます。
2. <strong>構造化された出力</strong> — `response_format` を渡すことで、エージェントは自由形式のテキストではなく
   検証済みの `TravelPlan` オブジェクトを返し、下流処理の信頼性を高めます。
3. <strong>計画の実行</strong> — コンシェルジュエージェントは、専門家ツール（`book_flight`, `reserve_hotel`, `book_activity`）を使って
   サブタスクを繰り返し処理し、計画を遂行して結果を報告します。

このパターンは、<em>何をするか</em>（計画）と <em>どうやってするか</em>（実行）を分離し、エージェントをより
モジュール化、テスト可能、かつ拡張しやすくします。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
